# 04 — Fine-tuning DistilBERT

Fine-tunes `distilbert-base-uncased` on the IMDB train/val splits from `02`,
evaluates on the held-out test set, and exports the model for the FastAPI service.

**Run on Google Colab with a GPU runtime** (Runtime → Change runtime type → T4).
The full training cell takes ~1 hour; everything else is quick. Colab-specific
cells (`google.colab` download) won't run locally — this notebook is the record
of *how* the model was produced.

## 1. Load the base model

`AutoModelForSequenceClassification` with `num_labels=2` puts a fresh, untrained
2-class head on the pretrained model. The `id2label` mapping is baked into the
saved config, so the model reports `"positive"`/`"negative"` instead of `LABEL_0`.

In [23]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "distilbert-base-uncased"
label2id = {"negative": 0, "positive": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2,
    id2label={0: "negative", 1: "positive"}, label2id=label2id,
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 2. Load data, encode labels, tokenize

Labels mapped to 0/1 and the column renamed to `labels` — the exact name the
Trainer looks for. Tokenizing with `truncation=True` (512-token cap) but **no
padding**; padding is handled dynamically per-batch by the data collator, which
is faster than padding everything to full length.

In [24]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_csv('train.csv')
val_df   = pd.read_csv('val.csv')

train_df['sentiment'] = train_df['sentiment'].map(label2id)
val_df['sentiment']   = val_df['sentiment'].map(label2id)

# rename the column to 'labels' — the name the Trainer looks for
train_dataset = Dataset.from_pandas(train_df).rename_column("sentiment", "labels")
val_dataset   = Dataset.from_pandas(val_df).rename_column("sentiment", "labels")

def tokenize(batch):
    return tokenizer(batch['review'], truncation=True)     # no padding here

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_val   = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/34706 [00:00<?, ? examples/s]

Map:   0%|          | 0/4959 [00:00<?, ? examples/s]

### Sanity check: labels made it in

Confirms `labels` exists and is an integer — this is the exact check that catches
the "model did not return a loss" error before it happens.

In [26]:
print(tokenized_train.column_names)      # must include 'labels'
print(tokenized_train[0]['labels'])      # must be 0 or 1

['review', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']
0


## 3. Smoke test — 1 epoch, 100 examples

Proves the whole loop runs end to end (tokenize → collate → train → eval → save)
in ~30 seconds before committing an hour of GPU to the full run. The resulting
model is meaningless here; we only care that nothing crashes.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)   # dynamic padding

# smoke-test subsets — these are what we actually train on for the dry run
small_train = tokenized_train.shuffle(seed=42).select(range(100))
small_val   = tokenized_val.shuffle(seed=42).select(range(100))

args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    eval_strategy='epoch',          
)

trainer = Trainer(                  
    model=model,
    args=args,
    train_dataset=small_train,      # ← smoke test uses the small sets
    eval_dataset=small_val,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.685589


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7, training_loss=0.6921024322509766, metrics={'train_runtime': 20.33, 'train_samples_per_second': 4.919, 'train_steps_per_second': 0.344, 'total_flos': 13246739865600.0, 'train_loss': 0.6921024322509766, 'epoch': 1.0})

## 4. Metrics function

Returns accuracy and F1 so the Trainer reports them each epoch, not just loss.

In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),   # binary F1, matches your baseline's macro-ish number
    }

## 5. Full fine-tuning run — 2 epochs, full dataset

Same wiring as the smoke test but on all ~34K training reviews, with
`compute_metrics` attached.

In [ ]:
args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    eval_strategy='epoch',          
)

trainer = Trainer(                  
    model=model,
    args=args,
    train_dataset=tokenized_train,      
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics = compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.225615,0.198224,0.928413,0.929745
2,0.113450,0.241186,0.934664,0.935252


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4340, training_loss=0.18793963067542574, metrics={'train_runtime': 4008.2296, 'train_samples_per_second': 17.317, 'train_steps_per_second': 1.083, 'total_flos': 9097308819716088.0, 'train_loss': 0.18793963067542574, 'epoch': 2.0})

### Reading the training curve

Training loss fell across both epochs (0.23 → 0.11), but **validation loss rose**
in epoch 2 (0.198 → 0.241) while F1 barely moved — the early signature of mild
overfitting. This is the evidence that **2 epochs is the sweet spot**: a third
would likely hurt generalization. Epochs chosen from the numbers, not guessed.

## 6. Final evaluation on the held-out test set

The reported number. Measured on the same `test.csv` the baseline used, so the
comparison is apples-to-apples.

In [30]:
test_df = pd.read_csv('test.csv')
test_df['sentiment'] = test_df['sentiment'].map(label2id)
test_dataset = Dataset.from_pandas(test_df).rename_column("sentiment", "labels")
tokenized_test = test_dataset.map(tokenize, batched=True)

test_metrics = trainer.evaluate(tokenized_test)
print(test_metrics)

Map:   0%|          | 0/9917 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.113450,0.241426,2,0.936674,0.937438


{'eval_loss': 0.24142581224441528, 'eval_accuracy': 0.9366743974992437, 'eval_f1': 0.9374377366009166}


### Result

Test F1 **0.937**, accuracy **0.937** — **+4.4 points** over the TF-IDF baseline
(0.893). Test ≈ validation, so the model generalized cleanly with no leakage.

In [31]:
import json
with open('results/distilbert_metrics.json', 'w') as f:
    json.dump({
        'model': 'distilbert',
        'accuracy': 0.9366743974992437,
        'f1': 0.9374377366009166,
    }, f, indent=2)

## 7. Save & export

Colab's disk is ephemeral, so we save the model + tokenizer and download the zip
before the runtime disconnects. Weights live outside git (gitignored) and are
loaded by the FastAPI service via a local path.

In [34]:
trainer.save_model('./distilbert-imdb')
tokenizer.save_pretrained('./distilbert-imdb')

!zip -r distilbert-imdb.zip distilbert-imdb

from google.colab import files
files.download('distilbert-imdb.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: distilbert-imdb/ (stored 0%)
  adding: distilbert-imdb/model.safetensors (deflated 8%)
  adding: distilbert-imdb/tokenizer_config.json (deflated 43%)
  adding: distilbert-imdb/config.json (deflated 51%)
  adding: distilbert-imdb/tokenizer.json (deflated 71%)
  adding: distilbert-imdb/training_args.bin (deflated 54%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>